# FMitF Transact Verification Factor Analysis

Generates synthetic .transact files, compiles via the FMitF compiler,
extracts verification-time statistics, and performs factor analysis.

**Run cells sequentially from top to bottom.**


In [41]:
import os, sys, json, subprocess, time, math, shutil
from pathlib import Path

import matplotlib
if 'get_ipython' in globals():
    matplotlib.use('module://ipykernel.pylab.backend_inline')
else:
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression

ROOT = Path('/Users/coned/Code/FMitF_rs')
GENERATOR = ROOT / 'examples' / 'gen_transact.py'
BASE_TMP = ROOT / 'tmp' / 'factor_sweep'
BASE_TMP.mkdir(parents=True, exist_ok=True)
(BASE_TMP / 'input').mkdir(exist_ok=True)
(BASE_TMP / 'output').mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 150, 'savefig.bbox': 'tight'})
print(f'Backend: {matplotlib.get_backend()}')
print(f'TMP  ={BASE_TMP}')


Backend: module://ipykernel.pylab.backend_inline
TMP  =/Users/coned/Code/FMitF_rs/tmp/factor_sweep


## 2. Parameter Grid

One-factor-at-a-time sweep over 31 configurations.


In [42]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class GenConfig:
    name: str
    num_transactions: int = 50
    tables_min: int = 2
    fields_per_table: Optional[int] = None
    pk_arity_min: int = 1
    pk_arity_max: int = 2
    hops_min: int = 1
    hops_max: int = 8
    ops_min: int = 1
    ops_max: int = 4
    ops_zipf_s: Optional[float] = None
    op_read_p: Optional[float] = None
    op_write_p: Optional[float] = None
    op_inc_dec_p: Optional[float] = None
    op_logical_p: Optional[float] = None
    for_loop_prob: float = 0.1
    for_loop_iters_min: int = 5
    for_loop_iters_max: int = 10
    if_prob: float = 0.25
    if_else_prob: float = 0.5
    if_body_min_ops: int = 1
    if_body_max_ops: int = 2
    continuation_prob: float = 0.3
    seed: int = 42

    def to_cli_args(self):
        cli_map = {
            'num_transactions': 'num-transactions',
            'tables_min': 'tables-min',
            'fields_per_table': 'fields-per-table',
            'pk_arity_min': 'pk-arity-min',
            'pk_arity_max': 'pk-arity-max',
            'hops_min': 'hops-min',
            'hops_max': 'hops-max',
            'ops_min': 'ops-min',
            'ops_max': 'ops-max',
            'ops_zipf_s': 'ops-zipf-s',
            'op_read_p': 'op-read-p',
            'op_write_p': 'op-write-p',
            'op_inc_dec_p': 'op-inc-dec-p',
            'op_logical_p': 'op-logical-p',
            'for_loop_prob': 'for-loop-prob',
            'for_loop_iters_min': 'for-loop-iters-min',
            'for_loop_iters_max': 'for-loop-iters-max',
            'if_prob': 'if-prob',
            'if_else_prob': 'if-else-prob',
            'if_body_min_ops': 'if-body-min-ops',
            'if_body_max_ops': 'if-body-max-ops',
            'continuation_prob': 'continuation-prob',
            'seed': 'seed',
        }
        args = []
        for field, flag in cli_map.items():
            val = getattr(self, field)
            if val is not None:
                args += ['--' + flag, str(val)]
        return args

grid = []
def add(name, **kw):
    cfg = GenConfig(name=name)
    for k, v in kw.items():
        setattr(cfg, k, v)
    grid.append(cfg)

# Factor 1: Transaction count
add('txn_5',   num_transactions=5, seed=42)
add('txn_20',  num_transactions=20, seed=42)
add('txn_50',  num_transactions=50, seed=42)
add('txn_100', num_transactions=100, seed=42)
add('txn_200', num_transactions=200, seed=42)
# Factor 2: Hops range
add('hops_1_3', hops_min=1, hops_max=3)
add('hops_3_6', hops_min=3, hops_max=6)
add('hops_5_10', hops_min=5, hops_max=10)
# Factor 3: If-statement probability
add('if_0',   if_prob=0.0)
add('if_10',  if_prob=0.1)
add('if_30',  if_prob=0.3)
add('if_50',  if_prob=0.5)
# Factor 4: For-loop probability
add('loop_0',   for_loop_prob=0.0)
add('loop_15',  for_loop_prob=0.15)
add('loop_30',  for_loop_prob=0.3)
add('loop_50',  for_loop_prob=0.5)
add('loop_80',  for_loop_prob=0.8)
# Factor 5: Ops per hop range
add('ops_1_2', ops_min=1, ops_max=2)
add('ops_1_4', ops_min=1, ops_max=4)
add('ops_2_6', ops_min=2, ops_max=6)
add('ops_3_8', ops_min=3, ops_max=8)
# Factor 6: If-body size
add('ifbody_1',  if_body_min_ops=1, if_body_max_ops=1)
add('ifbody_1_2', if_body_min_ops=1, if_body_max_ops=2)
add('ifbody_1_4', if_body_min_ops=1, if_body_max_ops=4)
add('ifbody_2_6', if_body_min_ops=2, if_body_max_ops=6)
# Factor 7: Loop iteration count
add('loopiter_3_5', for_loop_iters_min=3, for_loop_iters_max=5)
add('loopiter_5_10', for_loop_iters_min=5, for_loop_iters_max=10)
add('loopiter_10_20', for_loop_iters_min=10, for_loop_iters_max=20)

print(f'Grid: {len(grid)} configs')
for g in grid:
    print(f'  {g.name:>18s}: txns={g.num_transactions}')


Grid: 28 configs
               txn_5: txns=5
              txn_20: txns=20
              txn_50: txns=50
             txn_100: txns=100
             txn_200: txns=200
            hops_1_3: txns=50
            hops_3_6: txns=50
           hops_5_10: txns=50
                if_0: txns=50
               if_10: txns=50
               if_30: txns=50
               if_50: txns=50
              loop_0: txns=50
             loop_15: txns=50
             loop_30: txns=50
             loop_50: txns=50
             loop_80: txns=50
             ops_1_2: txns=50
             ops_1_4: txns=50
             ops_2_6: txns=50
             ops_3_8: txns=50
            ifbody_1: txns=50
          ifbody_1_2: txns=50
          ifbody_1_4: txns=50
          ifbody_2_6: txns=50
        loopiter_3_5: txns=50
       loopiter_5_10: txns=50
      loopiter_10_20: txns=50


## 3. Helper Functions

Generation, compilation, parsing, and summary extraction.


In [43]:
def generate_transact(cfg):
    out = BASE_TMP / 'input' / (cfg.name + '.transact')
    args = ['python3', str(GENERATOR)] + cfg.to_cli_args() + ['--out', str(out)]
    r = subprocess.run(args, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  [GEN FAIL] {cfg.name}: {r.stderr[:300]}')
    return out

def compile_transact(tpath, name):
    out_dir = BASE_TMP / 'output' / (name + '_out')
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    args = ['cargo', 'run', '--release', '--', str(tpath), str(out_dir), '--clear-cache']
    try:
        r = subprocess.run(args, capture_output=True, text=True, timeout=600, cwd=str(ROOT))
    except subprocess.TimeoutExpired:
        print(f'  [TIMEOUT] {name} (>600s)')
        return None
    dj = out_dir / 'data.json'
    if dj.exists():
        print(f'  [OK] {name} -> data.json')
        return dj
    else:
        print(f'  [FAIL] {name}')
        return None

def parse_header(path):
    info = {}
    if not path.exists():
        return info
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s.startswith('//') and '=' in s:
                try:
                    k, v = s.replace('//', '', 1).strip().split('=', 1)
                    v = v.strip()
                    for conv in [int, float]:
                        try:
                            v = conv(v)
                            break
                        except ValueError:
                            pass
                    info[k.strip()] = v
                except Exception:
                    pass
            elif s and not s.startswith('//') and not s.startswith('/*'):
                break
    return info

def extract_records(dj_path):
    with open(dj_path) as f:
        data = json.load(f)
    recs = []
    for entry in data.get('c_edge_verifications', []):
        r = {'cedge_' + k: v for k, v in entry.items()}
        recs.append(r)
    return recs, data.get('summary', {}), data.get('input_file', '')

def make_summary(recs):
    if not recs:
        return {'num_c_edges': 0, 'total_time_ms': 0, 'avg_time_ms': 0}

    def get_time(r):
        d = r.get('cedge_durations')
        if isinstance(d, dict):
            for sk in ['real_ms', 'total_ms', 'boogie_ms']:
                if sk in d:
                    try: return float(d[sk])
                    except: pass
        v = r.get('cedge_duration_ms')
        if isinstance(v, dict):
            for sk in ['real_ms', 'total_ms']:
                if sk in v:
                    try: return float(v[sk])
                    except: pass
        try: return float(v)
        except (ValueError, TypeError):
            return 0.0

    times = [get_time(r) for r in recs]
    pass_count = sum(1 for r in recs if r.get('cedge_result') == 'Pass')
    n = len(recs)
    st = sorted(times)
    avg_bc = sum(r.get('cedge_branch_count', 0) for r in recs) / n if n else 0
    pct_loop = sum(1 for r in recs if r.get('cedge_has_loop', False)) / n * 100 if n else 0
    sum_dr = sum(r.get('cedge_db_read_count', 0) for r in recs)
    sum_dw = sum(r.get('cedge_db_write_count', 0) for r in recs)

    return {
        'num_c_edges': n, 'num_pass': pass_count, 'num_error': n - pass_count,
        'commutative_pct': 100.0 * pass_count / n if n else 0,
        'total_time_ms': sum(times),
        'avg_time_ms': sum(times) / len(times) if times else 0,
        'median_time_ms': st[len(st)//2] if st else 0,
        'max_time_ms': max(times) if times else 0,
        'min_time_ms': min(times) if times else 0,
        'avg_branch_count': avg_bc,
        'max_branch_count': max(r.get('cedge_branch_count', 0) for r in recs) if recs else 0,
        'pct_with_loop': pct_loop,
        'avg_db_read': sum_dr / n if n else 0,
        'avg_db_write': sum_dw / n if n else 0,
        'sum_db_read': sum_dr, 'sum_db_write': sum_dw,
    }

print('Helpers loaded.')


Helpers loaded.


## 4. Run the Full Sweep

**~15-60 min** depending on hardware. Each config is generated, compiled, and results extracted.


In [44]:
# --- CHANGE THIS TO False TO SKIP THE SWEEP ---
RUN_SWEEP = True

all_file_summaries = []
all_cedge_records = []
compiled_count = 0
failed_gen = 0
failed_comp = 0

if RUN_SWEEP:
    print(f'\n=== Starting sweep: {len(grid)} configs ===\n')

    for i, cfg in enumerate(grid):
        name = cfg.name
        print(f'\n[{i+1}/{len(grid)}] {name}')

        tp = generate_transact(cfg)
        if not tp.exists():
            print(f'  SKIPPED (generation failed)')
            failed_gen += 1
            continue

        header = parse_header(tp)
        print(f'  Generated: {header.get("num_transactions","?")} txns, '
              f'{header.get("num_tables","?")} tables')

        dj = compile_transact(tp, name)
        if not dj:
            print(f'  SKIPPED (compilation failed)')
            failed_comp += 1
            continue

        compiled_count += 1
        recs, summary, inp = extract_records(dj)
        for r in recs:
            r['variant_name'] = name
        all_cedge_records.extend(recs)

        fs = make_summary(recs)
        for k, v in header.items():
            fs['config_' + k] = v
        fs['variant_name'] = name
        fs['compile_status'] = 'OK'
        all_file_summaries.append(fs)

        tc = fs.get('total_time_ms', 0)
        te = fs.get('num_c_edges', 0)
        ct = fs.get('commutative_pct', 0)
        print(f'  RESULT: {te} c-edges, {ct:.0f}% commutative, {tc:.0f}ms total')

    print(f'\n=== SWEEP COMPLETE ===')
    print(f'  Compiled: {compiled_count}/{len(grid)}  Errors: gen={failed_gen} comp={failed_comp}')
    print(f'  C-edge records: {len(all_cedge_records)}')

    with open(BASE_TMP / 'all_cedge_records.json', 'w') as f:
        json.dump(all_cedge_records, f)
    with open(BASE_TMP / 'all_file_summaries.json', 'w') as f:
        json.dump(all_file_summaries, f)
    print('Saved intermediate JSONs.')
else:
    print('=== SWEEP SKIPPED (RUN_SWEEP=False) ===')
    p1 = BASE_TMP / 'all_file_summaries.json'
    p2 = BASE_TMP / 'all_cedge_records.json'
    if p1.exists() and p2.exists():
        with open(p1) as f: all_file_summaries = json.load(f)
        with open(p2) as f: all_cedge_records = json.load(f)
        print(f'Loaded: {len(all_file_summaries)} files, {len(all_cedge_records)} c-edge records')
    else:
        print('No saved data. Run with RUN_SWEEP=True or place files in tmp/factor_sweep/')

print('Done.')



=== Starting sweep: 28 configs ===


[1/28] txn_5
  Generated: 5 txns, 3 tables
  [OK] txn_5 -> data.json
  RESULT: 2 c-edges, 100% commutative, 920ms total

[2/28] txn_20
  Generated: 20 txns, 5 tables
  [OK] txn_20 -> data.json
  RESULT: 118 c-edges, 24% commutative, 51212ms total

[3/28] txn_50
  Generated: 50 txns, 8 tables
  [OK] txn_50 -> data.json
  RESULT: 369 c-edges, 22% commutative, 153338ms total

[4/28] txn_100
  Generated: 100 txns, 10 tables
  [OK] txn_100 -> data.json
  RESULT: 1139 c-edges, 20% commutative, 1378540ms total

[5/28] txn_200
  Generated: 200 txns, 15 tables
  [OK] txn_200 -> data.json
  RESULT: 3446 c-edges, 21% commutative, 3114912ms total

[6/28] hops_1_3
  Generated: 50 txns, 8 tables
  [OK] hops_1_3 -> data.json
  RESULT: 305 c-edges, 33% commutative, 129167ms total

[7/28] hops_3_6
  Generated: 50 txns, 8 tables
  [OK] hops_3_6 -> data.json
  RESULT: 1325 c-edges, 24% commutative, 700673ms total

[8/28] hops_5_10
  Generated: 50 txns, 8 tables
  [OK

## 5. Load & Inspect Data


In [45]:
if not all_file_summaries:
    p1 = BASE_TMP / 'all_file_summaries.json'
    p2 = BASE_TMP / 'all_cedge_records.json'
    if p1.exists():
        with open(p1) as f: all_file_summaries = json.load(f)
    if p2.exists():
        with open(p2) as f: all_cedge_records = json.load(f)

df_files = pd.DataFrame(all_file_summaries)
df_cedge = pd.DataFrame(all_cedge_records)

for col in df_files.columns:
    if col.startswith('config_'):
        df_files[col] = pd.to_numeric(df_files[col], errors='coerce')

df_files = df_files.rename(columns={'variant_name': 'variant'})
print(f'Files: {len(df_files)} rows, C-edges: {len(df_cedge)} rows')

for col in ['total_time_ms', 'avg_time_ms', 'num_c_edges', 'commutative_pct']:
    if col in df_files.columns:
        v = df_files[col].dropna()
        if len(v) > 0:
            print(f'  {col:>20s}: mean={v.mean():.1f} median={v.median():.1f} '
                  f'min={v.min():.1f} max={v.max():.1f}')

df_files.to_csv(BASE_TMP / 'file_summary.csv', index=False)
df_cedge.to_csv(BASE_TMP / 'cedge_records.csv', index=False)

cols = [c for c in ['variant','num_c_edges','total_time_ms','avg_time_ms',
                     'commutative_pct','avg_branch_count','pct_with_loop']
        if c in df_files.columns]
if cols:
    df_files[cols].sort_values('total_time_ms', ascending=False).head(10)


Files: 28 rows, C-edges: 17960 rows
         total_time_ms: mean=420091.0 median=197520.3 min=920.0 max=3114912.5
           avg_time_ms: mean=574.3 median=544.3 min=415.5 max=1210.3
           num_c_edges: mean=641.4 median=369.0 min=2.0 max=3446.0
       commutative_pct: mean=26.9 median=23.0 min=11.9 max=100.0


## 6. Factor Analysis Figures

| Fig | File | What it shows |
|-----|------|---------------|
| 1 | fig01_commutativity.png | Commutativity (Pass vs Error) |
| 2 | fig02_branch_count.png | Branch count vs time |
| 3 | fig03_loops.png | Loop presence vs time |
| 4 | fig04_txn_count.png | Transaction count / SC-graph size |
| 5 | fig05_hops.png | Hops range vs time |
| 6 | fig06_loop_prob.png | For-loop probability vs time |
| 7 | fig07_if_prob.png | If-probability vs time/branches |
| 8 | fig08_db_access.png | DB read/write counts vs time |
| 9 | fig09_all_factors.png | All factors dashboard |
| 10 | fig10_correlation.png | Correlation matrix |
| 11 | fig11_regression.png | Multi-factor regression |


In [46]:
def extract_time(values):
    def to_float(v):
        if isinstance(v, (int, float)): return float(v)
        if isinstance(v, dict):
            for k in ['real_ms', 'total_ms', 'boogie_ms']:
                if k in v:
                    try: return float(v[k])
                    except: pass
        try: return float(v)
        except: return 0.0
    return [to_float(v) for v in values]


### Figure 1: Commutativity vs Verification Time


In [47]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

if 'cedge_duration_ms' in df_cedge.columns:
    df_cedge['time_ms'] = extract_time(df_cedge['cedge_duration_ms'])
    dv = df_cedge[df_cedge['time_ms'] > 0]

    ax = axes[0]
    for status in ['Pass', 'Error']:
        mask = dv['cedge_result'] == status
        if mask.any():
            ax.scatter(dv.loc[mask,'variant_name'], dv.loc[mask,'time_ms'],
                       label=status, alpha=0.6, s=40)
    ax.set_yscale('log')
    ax.set_xlabel('Variant'); ax.set_ylabel('Time (ms)')
    ax.set_title('Fig 1a: Commutativity per C-Edge')
    ax.legend(); ax.set_xticks([])

gp = df_files[['variant','commutative_pct','total_time_ms']].dropna()
gp['status'] = gp['commutative_pct'].apply(
    lambda x: 'Mostly Commutative' if x >= 50 else 'Mostly Not')
gp['status'] = pd.Categorical(gp['status'], categories=['Mostly Commutative','Mostly Not'])
sns.boxplot(data=gp, x='status', y='total_time_ms', ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_xlabel('Overall Commutativity')
axes[1].set_ylabel('Total Time (ms, log)')
axes[1].set_title('Fig 1b: Comm Rate vs Total Time')

for s in ['Mostly Commutative','Mostly Not']:
    sub = gp[gp['status'] == s]
    if len(sub) > 0:
        print(f'  {s:>20s}: n={len(sub)}, avg_time={sub["total_time_ms"].mean():.0f}ms')

fig.suptitle('Figure 1: Commutativity Factor', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(BASE_TMP / 'fig01_commutativity.png', dpi=150)
plt.show()
print('Saved: fig01_commutativity.png')


    Mostly Commutative: n=1, avg_time=920ms
            Mostly Not: n=27, avg_time=435616ms


<Figure size 2100x750 with 2 Axes>

<Figure size 2100x750 with 2 Axes>

<Figure size 2100x750 with 2 Axes>

<Figure size 2100x750 with 2 Axes>

...

<Figure size 3000x4800 with 32 Axes>

<Figure size 2100x2100 with 2 Axes>

<Figure size 1500x900 with 1 Axes>

<Figure size 2100x750 with 2 Axes>

Saved: fig01_commutativity.png


### Figure 2: Branch Count vs Time


In [48]:
bc = 'cedge_branch_count'
if bc in df_cedge.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14,5))

    file_info = df_files[['variant','total_time_ms']].rename(columns={'variant':'vn'})
    cedge_avg = df_cedge.groupby('variant_name').agg(
        {bc:'mean','cedge_duration_ms':'mean'}).reset_index()
    cedge_avg.columns = ['vn','avg_bc','avg_time']
    dbr = pd.merge(file_info, cedge_avg, on='vn', how='inner')

    ax = axes[0]
    for _, row in dbr.iterrows():
        ax.scatter(row['avg_bc'], row['avg_time'], s=100, edgecolors='black',
                   label=row['vn'], alpha=0.8)
    ax.set_xlabel('Avg Branch Count')
    ax.set_ylabel('Avg C-Edge Time (ms)')
    ax.set_yscale('log')
    ax.set_title('Fig 2a: Branch Count vs Time')
    ax.legend(fontsize=7, loc='upper left')

    ax2 = axes[1]
    bcv = df_files[['variant','avg_branch_count']].dropna().sort_values('avg_branch_count')
    sns.barplot(data=bcv, x='variant', y='avg_branch_count', ax=ax2, palette='viridis')
    ax2.set_xlabel('Variant'); ax2.set_ylabel('Avg Branch Count')
    ax2.set_title('Fig 2b: Avg Branch Count by Variant')
    ax2.tick_params(axis='x', rotation=45)

    if len(dbr) > 3:
        r, p = stats.pearsonr(dbr['avg_bc'], dbr['avg_time'])
        fig.text(0.5, 0.97, f'r={r:.3f}, p={p:.4e}', ha='center',
                 transform=fig.transFigure, fontsize=11)

    fig.suptitle('Figure 2: Branch Count Factor', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(BASE_TMP / 'fig02_branch_count.png', dpi=150)
    plt.show()
    print('Saved: fig02_branch_count.png')
else:
    print('No branch count data.')


/var/folders/kv/6dw98l5s6nb26lmvmgypsf1m0000gn/T/ipykernel_96434/3508403717.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=bcv, x='variant', y='avg_branch_count', ax=ax2, palette='viridis')


<Figure size 2100x750 with 2 Axes>

Saved: fig02_branch_count.png


### Figure 3: Loop Presence vs Time


In [49]:
hl = 'cedge_has_loop'
if hl in df_cedge.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14,5))

    lo = df_cedge.groupby(['variant_name', hl]).agg(
        {'cedge_duration_ms':'mean'}).reset_index()
    pv = lo.pivot(index='variant_name', columns=hl, values='cedge_duration_ms')
    pv = pv.reindex(columns=[False, True], fill_value=0)

    x = np.arange(len(pv))
    w = 0.35
    axes[0].bar(x - w/2, pv.get(False, 0), w, label='No Loop', alpha=0.8)
    axes[0].bar(x + w/2, pv.get(True, 0), w, label='Has Loop', alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(pv.index, rotation=45, ha='right')
    axes[0].set_ylabel('Avg C-Edge Time (ms)')
    axes[0].set_title('Fig 3a: Loop vs No-Loop')
    axes[0].legend()

    pl = df_cedge.groupby('variant_name')[hl].apply(
        lambda x: (x == True).mean() * 100).reset_index()
    pl.columns = ['vn', 'pct_loop']
    dlp = pd.merge(pl, df_files[['variant','total_time_ms']].rename(columns={'variant':'vn'}),
                   on='vn', how='left')

    dlp.plot.scatter('pct_loop', 'total_time_ms', s=80, ax=axes[1])
    for _, row in dlp.iterrows():
        axes[1].annotate(row['vn'], (row['pct_loop'], row['total_time_ms']),
                         fontsize=8, xytext=(5, 5), textcoords='offset points')
    axes[1].set_xlabel('% C-Edges with Loops')
    axes[1].set_ylabel('Total Time (ms)')
    axes[1].set_yscale('log')
    axes[1].set_title('Fig 3b: Loop % vs Total Time')

    if len(dlp) > 3:
        r, p = stats.pearsonr(dlp['pct_loop'], dlp['total_time_ms'])
        fig.text(0.5, 0.97, f'r={r:.3f}, p={p:.4e}', ha='center',
                 transform=fig.transFigure, fontsize=11)

    fig.suptitle('Figure 3: Loop Presence Factor', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(BASE_TMP / 'fig03_loops.png', dpi=150)
    plt.show()
    print('Saved: fig03_loops.png')
else:
    print('No loop data.')


<Figure size 2100x750 with 2 Axes>

Saved: fig03_loops.png


### Figure 4: Transaction Count (SC-graph Size) vs Time


In [50]:
tc_col = 'config_num_transactions'
if tc_col and tc_col in df_files.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18,5))
    ds = df_files.sort_values(tc_col)

    sc = axes[0].scatter(ds[tc_col], ds['total_time_ms'], s=80, alpha=0.7,
                         c=ds['commutative_pct'], cmap='RdYlGn', edgecolors='none')
    for _, row in ds.iterrows():
        axes[0].annotate(row['variant'], (row[tc_col], row['total_time_ms']),
                        fontsize=7, ha='center', va='bottom')
    axes[0].set_xlabel('Transaction Count')
    axes[0].set_ylabel('Total Time (ms, log)')
    axes[0].set_yscale('log')
    axes[0].set_title('Fig 4a: Txn vs Total Time')
    plt.colorbar(sc, ax=axes[0]).set_label('Comm %')

    sc2 = axes[1].scatter(ds[tc_col], ds['avg_time_ms'], s=80, alpha=0.7,
                          c=ds['num_c_edges'], cmap='Blues', edgecolors='none')
    for _, row in ds.iterrows():
        axes[1].annotate(row['variant'], (row[tc_col], row['avg_time_ms']),
                        fontsize=7, ha='center', va='bottom')
    axes[1].set_xlabel('Transaction Count')
    axes[1].set_ylabel('Avg C-Edge Time (ms, log)')
    axes[1].set_yscale('log')
    axes[1].set_title('Fig 4b: Txn vs Avg Time')
    plt.colorbar(sc2, ax=axes[1]).set_label('# C-Edges')

    bd = ds.sort_values('num_c_edges', ascending=False)
    sns.barplot(data=bd, x='variant', y='num_c_edges', ax=axes[2], palette='Blues_r')
    axes[2].set_xlabel('Variant'); axes[2].set_ylabel('# C-Edges')
    axes[2].set_title('Fig 4c: SC Graph Size')
    axes[2].tick_params(axis='x', rotation=45)

    fig.suptitle('Figure 4: Transaction Count Factor', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(BASE_TMP / 'fig04_txn_count.png', dpi=150)
    plt.show()
    print('Saved: fig04_txn_count.png')
else:
    print(f'Txn count missing. {[c for c in df_files.columns if "txn" in c.lower()]}')


/var/folders/kv/6dw98l5s6nb26lmvmgypsf1m0000gn/T/ipykernel_96434/4075567002.py:29: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=bd, x='variant', y='num_c_edges', ax=axes[2], palette='Blues_r')


<Figure size 2700x750 with 5 Axes>

Saved: fig04_txn_count.png


### Figure 5: Hops Range vs Time


In [51]:
hm = 'config_hops_max'
if hm and 'config_hops_min' in df_files.columns:
    fig, ax = plt.subplots(figsize=(10,6))
    dp = df_files.dropna(subset=[hm, 'config_hops_min'])
    dp['hops_range'] = dp[hm] - dp['config_hops_min']

    sc = ax.scatter(dp[hm], dp['total_time_ms'], s=100, alpha=0.7,
                   c=dp['hops_range'], cmap='viridis', edgecolors='none')
    for _, row in dp.iterrows():
        ax.annotate(row['variant'], (row[hm], row['total_time_ms']),
                   fontsize=8, ha='center', va='bottom')
    ax.set_xlabel('Max Hops')
    ax.set_ylabel('Total Time (ms, log)')
    ax.set_yscale('log')
    ax.set_title('Fig 5: Max Hops vs Time')
    plt.colorbar(sc, ax=ax).set_label('Hops Range')

    fig.tight_layout()
    fig.savefig(BASE_TMP / 'fig05_hops.png', dpi=150)
    plt.show()
    print('Saved: fig05_hops.png')
else:
    print(f'Hops cols missing. {[c for c in df_files.columns if "hops" in c.lower()]}')


<Figure size 1500x900 with 2 Axes>

Saved: fig05_hops.png


### Figure 6: For-Loop Probability vs Time


In [52]:
flp = 'config_for_loop_prob'
if flp and flp in df_files.columns:
    fig, ax = plt.subplots(figsize=(10,6))
    dp = df_files.dropna(subset=[flp])

    sc = ax.scatter(dp[flp], dp['total_time_ms'], s=100, alpha=0.7,
                   c=dp['pct_with_loop'], cmap='Reds', edgecolors='none')
    for _, row in dp.iterrows():
        ax.annotate(row['variant'], (row[flp], row['total_time_ms']),
                   fontsize=8, ha='center', va='bottom')
    ax.set_xlabel('For-Loop Probability')
    ax.set_ylabel('Total Time (ms, log)')
    ax.set_yscale('log')
    ax.set_title('Fig 6: For-Loop Prob vs Time')
    plt.colorbar(sc, ax=ax).set_label('% C-Edges with Loop')

    if len(dp) > 3:
        r, p = stats.pearsonr(dp[flp], np.log1p(dp['total_time_ms']))
        ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.4e}', transform=ax.transAxes)

    fig.tight_layout()
    fig.savefig(BASE_TMP / 'fig06_loop_prob.png', dpi=150)
    plt.show()
    print('Saved: fig06_loop_prob.png')
else:
    print(f'For-loop prob missing. {[c for c in df_files.columns if "loop" in c.lower()]}')


<Figure size 1500x900 with 2 Axes>

Saved: fig06_loop_prob.png


### Figure 7: If-Probability vs Time


In [53]:
ifp = 'config_if_prob'
if ifp and ifp in df_files.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    dp = df_files.dropna(subset=[ifp])

    sc = axes[0].scatter(dp[ifp], dp['total_time_ms'], s=100, alpha=0.7,
                        c=dp['avg_branch_count'], cmap='Oranges', edgecolors='none')
    for _, row in dp.iterrows():
        axes[0].annotate(row['variant'], (row[ifp], row['total_time_ms']),
                        fontsize=8, ha='center', va='bottom')
    axes[0].set_xlabel('If-Statement Probability')
    axes[0].set_ylabel('Total Time (ms, log)')
    axes[0].set_yscale('log')
    axes[0].set_title('Fig 7a: If-Prob vs Total Time')
    plt.colorbar(sc, ax=axes[0]).set_label('Avg Branch Count')

    bv = dp.sort_values(ifp)
    sns.barplot(data=bv, x=ifp, y='avg_branch_count', ax=axes[1], palette='Oranges')
    axes[1].set_xlabel('If-Probability')
    axes[1].set_ylabel('Avg Branch Count')
    axes[1].set_title('Fig 7b: If-Prob vs Branch Count')

    fig.suptitle('Figure 7: If-Probability Factor', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(BASE_TMP / 'fig07_if_prob.png', dpi=150)
    plt.show()
    print('Saved: fig07_if_prob.png')
else:
    print(f'If-prob missing. {[c for c in df_files.columns if "if" in c.lower()]}')


/var/folders/kv/6dw98l5s6nb26lmvmgypsf1m0000gn/T/ipykernel_96434/1528121134.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=bv, x=ifp, y='avg_branch_count', ax=axes[1], palette='Oranges')


<Figure size 2100x750 with 3 Axes>

Saved: fig07_if_prob.png


### Figure 8: DB Read/Write Counts vs Time


In [54]:
dr_col = 'cedge_db_read_count' in df_cedge.columns
dw_col = 'cedge_db_write_count' in df_cedge.columns

if dr_col and dw_col:
    fig, axes = plt.subplots(1, 3, figsize=(18,5))

    da = df_cedge.groupby('variant_name').agg({
        'cedge_db_read_count': ['sum','mean'],
        'cedge_db_write_count': ['sum','mean']
    }).reset_index()
    da.columns = ['vn', 'sr', 'ar', 'sw', 'aw']
    df2 = pd.merge(da, df_files[['variant','total_time_ms']].rename(columns={'variant':'vn'}),
                   on='vn')

    s0 = axes[0].scatter(df2['sr'], df2['total_time_ms'], s=80, alpha=0.7,
                        c=df2['sw'], cmap='Greens', edgecolors='none')
    for _, row in df2.iterrows():
        axes[0].annotate(row['vn'], (row['sr'], row['total_time_ms']),
                        fontsize=7, ha='center', va='bottom')
    axes[0].set_xlabel('Total DB Reads')
    axes[0].set_ylabel('Time (ms, log)')
    axes[0].set_yscale('log')
    axes[0].set_title('Fig 8a: DB Reads')
    if s0 is not None:
        plt.colorbar(s0, ax=axes[0]).set_label('Total Writes')

    s1 = axes[1].scatter(df2['sw'], df2['total_time_ms'], s=80, alpha=0.7,
                        c=df2['sr'], cmap='Blues', edgecolors='none')
    for _, row in df2.iterrows():
        axes[1].annotate(row['vn'], (row['sw'], row['total_time_ms']),
                        fontsize=7, ha='center', va='bottom')
    axes[1].set_xlabel('Total DB Writes')
    axes[1].set_ylabel('Time (ms, log)')
    axes[1].set_yscale('log')
    axes[1].set_title('Fig 8b: DB Writes')
    if s1 is not None:
        plt.colorbar(s1, ax=axes[1]).set_label('Total Reads')

    trw = df2['sr'] + df2['sw']
    axes[2].scatter(trw, df2['total_time_ms'], s=100, alpha=0.7, edgecolors='none')
    for i in range(len(df2)):
        row = df2.iloc[i]
        axes[2].annotate(row['vn'], (trw.iloc[i], row['total_time_ms']),
                        fontsize=7, ha='center', va='bottom')
    axes[2].set_xlabel('Total DB Ops (R+W)')
    axes[2].set_ylabel('Time (ms, log)')
    axes[2].set_yscale('log')
    axes[2].set_title('Fig 8c: Total DB Ops')

    fig.suptitle('Figure 8: DB Access Factor', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(BASE_TMP / 'fig08_db_access.png', dpi=150)
    plt.show()
    print('Saved: fig08_db_access.png')
else:
    print('DB data not available.')


<Figure size 2700x750 with 5 Axes>

Saved: fig08_db_access.png


### Figure 9: All Factors Dashboard


In [55]:
fcols = [c for c in df_files.columns if c.startswith('config_')]
extra = [c for c in ['pct_with_loop','avg_branch_count'] if c in df_files.columns]
allc = fcols + extra
n = len(allc)

if n > 0:
    nc = min(4, n); nr = math.ceil(n / nc)
    fig, axes = plt.subplots(nr, nc, figsize=(5*nc, 4*nr))
    axes_arr = np.array(axes).flatten()

    for i, col in enumerate(allc):
        ax = axes_arr[i]
        dp = df_files.dropna(subset=[col])
        if len(dp) > 2:
            x = dp[col].astype(float); y = dp['total_time_ms'].astype(float)
            ax.scatter(x, y, alpha=0.7, s=50, edgecolors='none')
            for _, row in dp.head(8).iterrows():
                ax.annotate(row['variant'], (row[col], row['total_time_ms']),
                           fontsize=7, xytext=(3, 3), textcoords='offset points')
            valid = ~(np.isnan(x.values) | np.isnan(y.values))
            if valid.sum() > 3:
                r, _ = stats.pearsonr(x.values[valid], np.log1p(y.values[valid]))
                ax.text(0.05, 0.95, f'r={r:.2f}', transform=ax.transAxes)
            ax.set_yscale('log')
            ax.set_title(col.replace('config_', ''))
            ax.tick_params(axis='x', rotation=45)
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
        ax.set_ylabel('Time (log)')

    for i in range(n, len(axes_arr)):
        axes_arr[i].axis('off')

    fig.suptitle('Figure 9: All Factors Dashboard', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(BASE_TMP / 'fig09_all_factors.png', dpi=150)
    plt.show()
    print('Saved: fig09_all_factors.png')
else:
    print('No factor columns.')


/var/folders/kv/6dw98l5s6nb26lmvmgypsf1m0000gn/T/ipykernel_96434/2227521583.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(x.values[valid], np.log1p(y.values[valid]))
/var/folders/kv/6dw98l5s6nb26lmvmgypsf1m0000gn/T/ipykernel_96434/2227521583.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(x.values[valid], np.log1p(y.values[valid]))
/var/folders/kv/6dw98l5s6nb26lmvmgypsf1m0000gn/T/ipykernel_96434/2227521583.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(x.values[valid], np.log1p(y.values[valid]))
/var/folders/kv/6dw98l5s6nb26lmvmgypsf1m0000gn/T/ipykernel_96434/2227521583.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(x.values[valid], np.log1p(y.values[valid]))
/var/folders/kv/6dw9

<Figure size 3000x4800 with 32 Axes>

Saved: fig09_all_factors.png


### Figure 10: Correlation Matrix


In [56]:
fc = [c for c in df_files.columns if (c.startswith('config_') or
    c in ['num_c_edges','total_time_ms','avg_time_ms','median_time_ms',
          'commutative_pct','avg_branch_count','pct_with_loop',
          'sum_db_read','sum_db_write'])]

ndf = df_files[fc].copy()
for c in fc: ndf[c] = pd.to_numeric(ndf[c], errors='coerce')
ndf = ndf.dropna(how='all')

cn = [c.replace('config_','') if c.startswith('config_') else c for c in ndf.columns]
ndf.columns = cn

cm = ndf.corr()
fig, ax = plt.subplots(figsize=(min(14,len(cn)), min(14,len(cn))))
mask = np.triu(np.ones_like(cm, dtype=bool))

sns.heatmap(cm, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
           vmin=-1, vmax=1, center=0, square=True, linewidths=1, ax=ax,
           cbar_kws={'shrink': 0.8})
ax.set_title('Fig 10: Factor Correlation Matrix', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
fig.tight_layout()
fig.savefig(BASE_TMP / 'fig10_correlation.png', dpi=150)
plt.show()
print('Saved: fig10_correlation.png')

if 'total_time_ms' in cm.columns:
    print('\nStrongest correlations with total_time_ms:')
    tc = cm['total_time_ms'].drop('total_time_ms').sort_values(ascending=False)
    for name, val in tc.items():
        if name != 'total_time_ms':
            print(f'  {name:>25s}: {val:+.3f}')


<Figure size 2100x2100 with 2 Axes>

Saved: fig10_correlation.png

Strongest correlations with total_time_ms:
                num_c_edges: +0.947
               sum_db_write: +0.908
                sum_db_read: +0.880
           num_transactions: +0.860
                 num_tables: +0.759
                avg_time_ms: +0.630
                   hops_min: +0.364
                   hops_max: +0.171
                    ops_max: +0.164
                    ops_min: +0.137
                    if_prob: +0.055
           avg_branch_count: +0.023
         for_loop_iters_max: -0.029
              pct_with_loop: -0.034
         for_loop_iters_min: -0.036
            if_body_min_ops: -0.065
            if_body_max_ops: -0.069
              for_loop_prob: -0.128
             median_time_ms: -0.135
            commutative_pct: -0.216
                           : +nan
                 tables_min: +nan
           fields_per_table: +nan
               pk_arity_min: +nan
               pk_arity_max: +nan
                  hops_dist: +nan
   

### Figure 11: Multi-Factor Regression


In [57]:
rcols = ['config_num_transactions','config_hops_max','config_ops_max',
         'config_for_loop_prob','config_if_prob']
rcols = [c for c in rcols if c in df_files.columns]

rdf = df_files[rcols + ['total_time_ms']].copy()
for c in rcols: rdf[c] = pd.to_numeric(rdf[c], errors='coerce')
rdf['total_time_ms'] = pd.to_numeric(rdf['total_time_ms'], errors='coerce')
rdf = rdf.dropna()

if len(rdf) > 5:
    X = rdf[rcols]; y_log = np.log1p(rdf['total_time_ms'])
    model = LinearRegression(); model.fit(X, y_log)
    r2 = model.score(X, y_log)

    fnames = [c.replace('config_','') for c in rcols]
    coeffs = dict(zip(fnames, model.coef_))

    fig, ax = plt.subplots(figsize=(10,6))
    si = sorted(coeffs.items(), key=lambda x: abs(x[1]), reverse=True)
    names, values = zip(*si)
    colors = ['red' if v < 0 else 'green' for v in values]
    bars = ax.barh(names[::-1], values[::-1], color=colors, alpha=0.8)
    ax.set_xlabel('Coefficient (log-time scale)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    for bar, val in zip(bars, values[::-1]):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
               f'{val:+.4f}', va='center')
    ax.set_title(f'Fig 11: Regression (R2={r2:.3f})')
    fig.tight_layout()
    fig.savefig(BASE_TMP / 'fig11_regression.png', dpi=150)
    plt.show()
    print('Saved: fig11_regression.png')

    print(f'\nR2 = {r2:.4f}')
    print(f'Intercept = {model.intercept_:.4f}')
    for name, val in si:
        impact = 'increases time' if val > 0 else 'decreases time'
        print(f'  {name:>25s}: {val:+.4f} ({impact})')
else:
    print(f'Not enough data for regression: {len(rdf)} rows')


<Figure size 1500x900 with 1 Axes>

Saved: fig11_regression.png

R2 = 0.5219
Intercept = 7.6303
                    if_prob: +2.3317 (increases time)
              for_loop_prob: -0.5179 (decreases time)
                    ops_max: +0.4230 (increases time)
                   hops_max: +0.0946 (increases time)
           num_transactions: +0.0290 (increases time)


## Summary

### Output Files (in `tmp/factor_sweep/`)

| File | Description |
|------|-------------|
| `fig01_commutativity.png` | Commutativity vs time |
| `fig02_branch_count.png` | Branch count vs time |
| `fig03_loops.png` | Loop presence vs time |
| `fig04_txn_count.png` | Transaction count / SC-graph size |
| `fig05_hops.png` | Hops range vs time |
| `fig06_loop_prob.png` | For-loop probability vs time |
| `fig07_if_prob.png` | If-probability vs time/branches |
| `fig08_db_access.png` | DB read/write counts vs time |
| `fig09_all_factors.png` | All factors dashboard |
| `fig10_correlation.png` | Correlation matrix heatmap |
| `fig11_regression.png` | Multi-factor regression coefficients |
| `all_cedge_records.json` | Raw per-c-edge data |
| `all_file_summaries.json` | Per-file summary data |
| `file_summary.csv` | CSV of file summaries |
| `cedge_records.csv` | CSV of c-edge records |

### Key Factors Expected

- **Transaction count**: Controls SC-graph size O(n^2 c-edges) — strongest time correlation
- **Branch count**: From if-statements; more branches = harder Boogie proofs
- **Loop presence**: Circularity = infinite-state reasoning — may increase time
- **For-loop probability**: Controls loop frequency in generated programs
- **If-probability**: Controls branch frequency → Boogie complexity
- **Commutativity (Pass vs Error)**: Whether each c-edge verified as commutative
